In [4]:
# directories
pdfs_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\pdfs'
ocr_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\ocr_text'
extracted_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\extracted_csv'

In [5]:
# post 1877 data extraction

import re
import csv
from typing import List, Dict, Tuple, Optional

# US states and territories (1877 era)
US_STATES = {
    'ALABAMA', 'ALASKA', 'ARIZONA', 'ARKANSAS', 'CALIFORNIA', 'COLORADO',
    'CONNECTICUT', 'DELAWARE', 'DISTRICT OF COLUMBIA', 'FLORIDA', 'GEORGIA',
    'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LOUISIANA',
    'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA', 'MISSISSIPPI',
    'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA', 'NEW HAMPSHIRE', 'NEW JERSEY',
    'NEW MEXICO', 'NEW YORK', 'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO',
    'OKLAHOMA', 'OREGON', 'PENNSYLVANIA', 'RHODE ISLAND', 'SOUTH CAROLINA',
    'SOUTH DAKOTA', 'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGINIA',
    'WASHINGTON', 'WEST VIRGINIA', 'WISCONSIN', 'WYOMING', 'DAKOTA',
    'INDIAN TERRITORY', 'MONTANA TERRITORY', 'NEW MEXICO TERRITORY',
    'UTAH TERRITORY', 'WASHINGTON TERRITORY', 'WYOMING TERRITORY',
    'ONTARIO', 'QUEBEC', 'NOVA SCOTIA', 'NEW BRUNSWICK', 'MANITOBA',
    'BRITISH COLUMBIA', 'PRINCE EDWARD ISLAND', 'NEWFOUNDLAND',
}


def normalize_state(state: str) -> str:
    """Normalize state names (handle OCR issues)."""
    replacements = {'Μ': 'M', 'Α': 'A', 'Ε': 'E', 'Ο': 'O', 'Ι': 'I', 
                    'Îœ': 'M', 'Î': 'A', 'œ': 'M', 'Ñ€Ð¾Ñ€': 'pop'}
    for old, new in replacements.items():
        state = state.replace(old, new)
    state = ''.join(c for c in state if ord(c) < 128 or c.isalpha())
    return state.strip().rstrip('. ').strip()


def is_state_header(line: str) -> bool:
    """Check if a line is a state header."""
    line = line.strip()
    if not line or len(line) < 4:
        return False
    cleaned = normalize_state(line)
    alpha_chars = [c for c in cleaned if c.isalpha()]
    if not alpha_chars or len(alpha_chars) < 4:
        return False
    if sum(1 for c in alpha_chars if c.isupper()) / len(alpha_chars) < 0.8:
        return False
    return cleaned in US_STATES


def is_town_line(line: str) -> bool:
    """Check if line starts a town entry."""
    line = line.strip()
    if not line:
        return False
    
    # Normalize "ST . " to "ST. " (OCR artifact)
    line = re.sub(r'^(ST|MT|FT|PT)\s+\.', r'\1.', line)

    # Normalize spaced "c . h ." to "c.h." (1890 OCR artifact for county seat)
    line = re.sub(r',\s*([cC])\s*\.\s*([hH])\s*\.', r', \1.\2.', line)
    
    if not re.match(r'^(?:(?:ST|MT|FT|PT)\s?\.| EL|LA|LE|DE|[A-Z]{3})', line):
        return False
    
    # Town: all caps, may include hyphens/periods, AND may have space-separated words
    town = r'[A-Z][A-Z\-\.]*(?:\s+[A-Z][A-Z\-\.]*)*'
    
    # County: Initial cap word(s) like "Mariposa" or "Contra Costa"
    county = r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*'
    
    patterns = [
        # === County seat patterns (C. H.) ===
        rf'^{town}\s?,\s*C\.\s*H\.\s*,',
        rf'^{town}\s?,\s*C\.H\.\s*,',
        rf'^{town}\s?,\s*C\.\s*H\.\s*,\s*{county}\s+Co\.?\s*[,\s]',
        rf'^{town}\s?,\s*C\.H\.\s*,\s*{county}\s+Co\.?\s*[,\s]',
        rf'^{town}\s?,\s*C\.\s*H\.\s*$',
        rf'^{town}\s?,\s*C\.H\.\s*$',
        rf'^{town}\s?,\s*c\.\s*h\.\s*,\s*{county}\s+Co\.?\s*[,\s]',
        
        # === Standard "County Co." patterns ===
        rf'^{town}\s?,\s*{county}\s+Co\.\s*,',
        rf'^{town}\s?,\s*{county}\s+Co\.\s*$',
        rf'^{town}\s?,\s*{county}\s+Co\.\s*;',
        rf'^{town}\s?,\s*{county}\s+Co\s*,',
        rf'^{town}\s?,\s*{county}\s+Co\s*$',
        rf'^{town}\s?,\s*{county}\s+Co\.?\s*,?\s*[\d,]+.*pop',
        
        # === OCR variation patterns ===
        rf'^{town}\s?,\s*{county}\s+Co\.\s+,',
        rf'^{town}\s?,\s*C\.\s*H\.\s*,\s*{county}\s+Co\.\s+,',
        rf'^{town}\s?,\s*{county}\s+Co\.\s*\d',
        rf'^{town}\s?,\s*{county}\s+Co,\.',
        rf'^{town}\s?,\s*C\.\s*H\.\s*,\s*{county}\s+Co,\.',
        rf'^{town}\s?,\s*{county}\s+Co\.\s*,\s*[a-z]',
    ]
    
    for pattern in patterns:
        if re.match(pattern, line):
            return True
    return False


def extract_town_name(line: str) -> str:
    """Extract town name from the start of a town line."""
    # Normalize "ST . " to "ST. " at start (OCR artifact)
    line = re.sub(r'^(ST|MT|FT|PT)\s+\.', r'\1.', line)

    match = re.match(r'^([A-Z][A-Z\s\-\.]+?)\s?,', line)
    return match.group(1).strip() if match else ""


def find_newspapers_in_block(block_text: str, town_name: str) -> List[Tuple[str, str]]:
    """Find all newspaper entries in a block of text."""
    newspapers = []
    
    # Frequency words
    freq_words = r'(?:every\s*(?:morning|evening|afternoon|day)|Sundays?|Mondays?|Tuesdays?|Wednesdays?|Thursdays?|Fridays?|Saturdays?|daily|weekly|semi-weekly|tri-weekly|monthly|bi-monthly|semi-monthly|quarterly)'
    
    # Political/type affiliations
    political_words = r'(?:democratic|republican|independent|neutral|liberal|conservative|greenback|prohibition|baptist|congregational|methodist|universalist|religious|agricultural|literary|german|french|spanish|comic)'
    
    # Combined indicator pattern
    indicator_pattern = rf'(?:{freq_words}|{political_words})'
    
    # Find all potential newspaper names (ALL CAPS sequences)
    name_pattern = r'([A-Z][A-Z\s\-\&\.\']+[A-Z])\s*[;:]'
    
    valid_newspapers = []
    
    for name_match in re.finditer(name_pattern, block_text):
        name = name_match.group(1).strip()
        name_end = name_match.end()
        
        if name == town_name:
            continue
        
        clean_name = name.replace('.', '').replace(' ', '').replace('-', '').replace('&', '').replace("'", '')
        if len(clean_name) < 3:
            continue
        if name.strip('. ') in ['CO', 'RD', 'POP', 'THE', 'AND', 'FOR', 'N', 'S', 'E', 'W', 'C', 'H']:
            continue
        if name.strip('. ') in US_STATES:
            continue
        
        lookahead_text = block_text[name_end:name_end + 200]
        
        next_caps_match = re.search(r'[;:]\s*([A-Z][A-Z\s\-\&\.\']{2,}[A-Z])\s*[;:]', lookahead_text)
        if next_caps_match:
            lookahead_text = lookahead_text[:next_caps_match.start()]
        
        sections = re.split(r'\s*;\s*', lookahead_text)[:3]
        lookahead_limited = ' ; '.join(sections)
        
        if re.search(indicator_pattern, lookahead_limited, re.IGNORECASE):
            valid_newspapers.append((name_match.start(), name_end, name))
    
    for i, (name_start, name_end, name) in enumerate(valid_newspapers):
        if i + 1 < len(valid_newspapers):
            entry_end = valid_newspapers[i + 1][0]
        else:
            entry_end = len(block_text)
        
        entry_text = block_text[name_start:entry_end].strip()
        entry_text = re.sub(r'\s+[A-Z]{3,}[A-Z\s\-\.]*,\s*(?:C\.\s*H\.|[A-Z][a-z]+\s+Co).*$', '', entry_text, flags=re.DOTALL)
        
        newspapers.append((name, entry_text))
    
    suspended_pattern = r'([A-Z][A-Z\s\-\&\.\']+[A-Z])\s*[\.:]?\s*(?:††|â€\s*â€|‡‡|\.\s*â€)'
    for match in re.finditer(suspended_pattern, block_text):
        name = match.group(1).strip()
        clean_name = name.replace('.', '').replace(' ', '')
        if name == town_name or len(clean_name) < 3:
            continue
        if name.strip('. ') in US_STATES:
            continue
        if not any(n[0] == name for n in newspapers):
            newspapers.append((name, f"{name} †† (suspended publication)"))
    
    return newspapers


def extract_frequency(text: str) -> str:
    """Extract publication frequency."""
    freq_map = [
        (r'every\s*morning', 'Daily'),
        (r'every\s*evening', 'Daily'),
        (r'every\s*afternoon', 'Daily'),
        (r'every\s*day', 'Daily'),
        (r'semi-weekly', 'Semi-weekly'), (r'tri-weekly', 'Tri-weekly'),
        (r'bi-weekly', 'Bi-weekly'), (r'bi-monthly', 'Bi-monthly'),
        (r'semi-month', 'Semi-monthly'),
        (r'sundays?', 'Sundays'), (r'mondays?', 'Mondays'), (r'tuesdays?', 'Tuesdays'),
        (r'wednesdays?', 'Wednesdays'), (r'thursdays?', 'Thursdays'),
        (r'fridays?', 'Fridays'), (r'saturdays?', 'Saturdays'),
        (r'\bdaily\b', 'Daily'), (r'\bweekly\b', 'Weekly'),
        (r'\bmonthly\b', 'Monthly'), (r'\bquarterly\b', 'Quarterly'),
    ]
    text_lower = text.lower()
    for pattern, freq in freq_map:
        if re.search(pattern, text_lower):
            return freq
    return ""


def extract_political(text: str) -> str:
    """Extract political affiliation."""
    affil_map = [
        (r'\bdemocrat', 'Democratic'), (r'\brepublican', 'Republican'),
        (r'\bindependent', 'Independent'), (r'\bneutral', 'Neutral'),
        (r'\bliberal', 'Liberal'), (r'\bconservative', 'Conservative'),
        (r'\bgreenback', 'Greenback'), (r'\bprohibition', 'Prohibition'),
        (r'\bbaptist', 'Baptist'), (r'\bcongregational', 'Congregational'),
        (r'\bmethodist', 'Methodist'), (r'\buniversalist', 'Universalist'),
        (r'\breligious', 'Religious'), (r'\bagricultural', 'Agricultural'),
        (r'\bliterary', 'Literary'), (r'\bgerman', 'German'),
        (r'\bcomic', 'Comic'),
    ]
    text_lower = text.lower()
    for pattern, affil in affil_map:
        if re.search(pattern, text_lower):
            return affil
    return ""


def extract_price(text: str) -> str:
    """Extract subscription price."""
    patterns = [
        r'subscription\s*\$\s*(\d+)\s+(\d+)',
        r'subscription\s*\$\s*(\d+\.\d+)',
        r'subscription\s*\$\s*(\d+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text.lower())
        if match:
            groups = match.groups()
            if len(groups) == 2:
                return f"${groups[0]}.{groups[1]}"
            return f"${groups[0]}"
    return ""


def extract_established(text: str) -> str:
    """Extract year established."""
    patterns = [
        r'estab-?\s*lished\s*(\d{4})',
        r'established\s*(\d{4})',
        r're-established\s*(\d{4})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text.lower())
        if match:
            return match.group(1)
    return ""


def extract_editor_publisher(text: str) -> Tuple[str, str]:
    """Extract editor and publisher names."""
    editor = ""
    publisher = ""
    
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?\s*and\s+pub[-\s]*l[-\s]*i[-\s]*s[-\s]*h[-\s]*e[-\s]*r[-\s]*s?', text)
    if match:
        name = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
        return name, name
    
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?\s*and\s+pro[-\s]*p[-\s]*r[-\s]*i[-\s]*e[-\s]*t[-\s]*o[-\s]*r[-\s]*s?', text)
    if match:
        name = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
        return name, name
    
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?[;,:\s]', text)
    if match:
        editor = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+(?:\s+Co\.)?),?\s*pub[-\s]*l[-\s]*i[-\s]*s[-\s]*h[-\s]*e[-\s]*r[-\s]*s?[;,:\s\.]', text)
    if match:
        publisher = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    if not publisher:
        match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+(?:\s+Co\.)?),?\s*pro[-\s]*p[-\s]*r[-\s]*i[-\s]*e[-\s]*t[-\s]*o[-\s]*r[-\s]*s?[;,:\s\.]', text)
        if match:
            publisher = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    return editor, publisher


def extract_circulation_early(text: str) -> str:
    """Extract circulation number for 1877-1878 (numeric values)."""
    # Normalize line-break hyphens first
    text = re.sub(r'-\s+', '', text)
    
    # Pattern for circulation with numeric values
    pattern = r"circ?(?:ulation|'n|e'n)[\s\-]*(?:about|approximately|nearly|over|around)?[\s\-]*(?:\w+\s+)?(\d[\d,]*)"
    
    # Also check for "claims XXX" pattern
    claims_pattern = r"claims[\s\-]+(\d[\d,]*)"
    
    match = re.search(pattern, text.lower())
    if not match:
        match = re.search(claims_pattern, text.lower())
    
    if match:
        circ = match.group(1).replace(',', '')
        if 'estimated' in text.lower() or "est'd" in text.lower():
            return f"{circ} (estimated)"
        return circ
    return ""


CIRCULATION_RATINGS = {
    "A1": 100000, "A": 75000,
    "B1": 50000, "B": 37500,
    "C1": 25000, "C": 22500,
    "D1": 20000, "D": 17500,
    "E1": 15000, "E": 12500,
    "F1": 10000, "F": 7500,
    "G1": 5000, "G": 4000,
    "H1": 3000, "H": 2500,
    "I1": 2000, "I": 1500,
    "J1": 1000, "J": 750,
    "K1": 500, "K": 250,
    "X1": None,
}


def extract_circulation_late(text: str) -> Optional[int]:
    """Extract circulation for 1879+ (letter rating codes)."""
    # First try letter codes (e.g., "circulation K 1", "circulationK", "circulation G")
    match = re.search(r"circ(?:ulation|'n)\s*([A-KX])\s*([1])?", text, re.IGNORECASE)
    if match:
        letter = match.group(1).upper()
        suffix = match.group(2) or ""
        code = letter + suffix
        return CIRCULATION_RATINGS.get(code)
    
    # Then try numeric values (e.g., "circulation 350", "circ'n 800")
    match = re.search(r"circ(?:ulation|'n)\s*(\d[\d,]*)", text.lower())
    if match:
        circ = match.group(1).replace(',', '')
        return int(circ)
    
    return None


def parse_newspaper_entry(name: str, text: str, state: str, town: str, year: int) -> Dict:
    """Parse a newspaper entry and return structured data."""
    editor, publisher = extract_editor_publisher(text)
    
    # Use different circulation extraction based on year
    if year <= 1878:
        circulation = extract_circulation_early(text)
    else:
        circulation = extract_circulation_late(text)
    
    return {
        'state': state,
        'town': town,
        'newspaper_name': name,
        'frequency': extract_frequency(text),
        'political_affiliation': extract_political(text),
        'subscription_price': extract_price(text),
        'established': extract_established(text),
        'editor': editor,
        'publisher': publisher,
        'circulation': circulation,
        'raw_text': re.sub(r'\s+', ' ', text).strip()
    }


def preprocess_text(content: str) -> str:
    """Preprocess text to handle formatting issues."""
    content = re.sub(r'---\s*Page\s*\d+\s*---', '\n<<PAGE_BREAK>>\n', content)
    
    removals = [
        r'^GEO\s*\.\s*P\.?\s*ROWELL.*$',
        r'^P\.\s*ROWELL\s*&.*$',
        r'^AMERICAN NEWSPAPER DIRECTORY\s*\.?\s*$',
        r'^LIBRARY\s*$', r'^UNIVERSITY\s+OF\s*$',
        r'^EXPLANATIONS\s*\.?\s*$', r'^POPULATION\s*\.?\s*$',
        r'^CIRCULATION\s*\.?\s*$', r'^ITALIC WORDS\s*\.?\s*$',
        r'^\d+\s*$', r"^CO'S\s*$", r'^GEO\s*\.\s*$',
    ]
    for pattern in removals:
        content = re.sub(pattern, '', content, flags=re.MULTILINE)
    
    for state in US_STATES:
        pattern = r'(<<PAGE_BREAK>>)\s*\n?\s*' + re.escape(state) + r'\s*\.?\s*$'
        content = re.sub(pattern, r'\1', content, flags=re.MULTILINE | re.IGNORECASE)
    
    content = re.sub(r'<<PAGE_BREAK>>', '\n', content)
    content = re.sub(r'A LIST,\s*ARRANGED ALPHABETICALLY.*?ETC\.', '', content, flags=re.DOTALL)
    return content


def extract_newspapers(content: str, year: int) -> List[Dict]:
    """Main extraction function."""
    content = preprocess_text(content)
    lines = content.split('\n')
    
    results = []
    current_state = ""
    i = 0
    
    while i < len(lines):
        line = lines[i].strip()
        
        if not line:
            i += 1
            continue
        
        if is_state_header(line):
            current_state = normalize_state(line)
            i += 1
            continue
        
        if current_state and is_town_line(line):
            town_name = extract_town_name(line)
            if not town_name:
                i += 1
                continue
            
            block_lines = [line]
            j = i + 1
            while j < len(lines):
                next_line = lines[j].strip()
                if not next_line:
                    block_lines.append('')
                    j += 1
                    continue
                
                if is_state_header(next_line):
                    normalized_next = normalize_state(next_line)
                    if normalized_next == current_state:
                        j += 1
                        continue
                    else:
                        break
                
                if is_town_line(next_line):
                    break
                    
                block_lines.append(next_line)
                j += 1
            
            block_text = ' '.join(block_lines)
            block_text = re.sub(r'-\s+', '', block_text)
            block_text = re.sub(r'\s+', ' ', block_text)
            
            newspapers = find_newspapers_in_block(block_text, town_name)
            
            for name, entry_text in newspapers:
                entry = parse_newspaper_entry(name, entry_text, current_state, town_name, year)
                results.append(entry)
            
            i = j
            continue
        
        i += 1
    
    return results


def print_state_counts(newspapers: List[Dict]):
    """Print the number of newspaper entries found for each state."""
    state_counts = {}
    for entry in newspapers:
        state = entry['state']
        state_counts[state] = state_counts.get(state, 0) + 1
    
    print("\n=== Newspaper Entries by State ===")
    for state in sorted(state_counts.keys()):
        print(f"  {state}: {state_counts[state]}")
    print(f"  {'─' * 30}")
    print(f"  TOTAL: {len(newspapers)}\n")


def main(input_file: str, output_file: str, year: int):
    """Process input file and write CSV output."""
    with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    newspapers = extract_newspapers(content, year)
    
    fieldnames = ['state', 'town', 'newspaper_name', 'frequency', 'political_affiliation',
                  'subscription_price', 'established', 'editor', 'publisher', 'circulation', 'raw_text']
    
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(newspapers)
    
    print(f"Extracted {len(newspapers)} newspaper entries to {output_file}")
    print_state_counts(newspapers)
    return newspapers


def output_comparison_file(input_file: str, output_file: str, comparison_file: str):
    """Create a comparison file with chars 15000-35000 of input and output."""
    with open(input_file, 'r', encoding='utf-8', errors='ignore') as f:
        f.seek(40000)
        input_content = f.read(20000)
    
    with open(output_file, 'r', encoding='utf-8') as f:
        f.seek(37000)
        output_content = f.read(20000)
    
    # with open(comparison_file, 'w', encoding='utf-8') as f:
    #     f.write("=== INPUT FILE (chars 15000-35000) ===\n\n")
    #     f.write(input_content)
    #     f.write("\n\n=== OUTPUT FILE (chars 15000-35000) ===\n\n")
    #     f.write(output_content)
    
    # print(f"Comparison file written to {comparison_file}")


import glob
import os

if __name__ == "__main__":
    input_folder = ocr_text_dir
    output_folder = extracted_text_dir
    
    # Find all files matching the pattern with v13 in the name
    pattern = os.path.join(input_folder, 'Rowell * - v13.txt')
    input_files = glob.glob(pattern)
    
    for input_file in input_files:
        # Extract the year from the filename
        filename = os.path.basename(input_file)
        match = re.search(r'Rowell (\d{4}) - v13\.txt', filename)
        
        if match:
            year = int(match.group(1))
            
            # Only process files from 1877 onwards
            if year < 1877:
                continue
            
            output_file = os.path.join(output_folder, f'Rowell {year}.csv')
            comparison_file = os.path.join(output_folder, f'Rowell {year}_comparison.txt')
            
            print(f"Processing: {input_file} (Year: {year})")
            print(f"  -> Output: {output_file}")
            print(f"  -> Comparison: {comparison_file}")
            
            main(input_file, output_file, year)
            # output_comparison_file(input_file, output_file, comparison_file)
            
            print(f"  Done!\n")

    print(f"Processed {len(input_files)} files.")

Processing: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1877 - v13.txt (Year: 1877)
  -> Output: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\extracted_csv\Rowell 1877.csv
  -> Comparison: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\extracted_csv\Rowell 1877_comparison.txt
Extracted 7274 newspaper entries to C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\extracted_csv\Rowell 1877.csv

=== Newspaper Entries by State ===
  ALABAMA: 81
  ARIZONA: 3
  ARKANSAS: 60
  CALIFORNIA: 218
  COLORADO: 41
  CONNECTICUT: 100
  DAKOTA: 15
  DELAWARE: 39
  DISTRICT OF COLUMBIA: 25
  FLORIDA: 26
  GEORGIA: 123
  IDAHO: 9
  ILLINOIS: 649
  INDIANA: 360
  IOWA: 330
  KANSAS: 159
  KENTUCKY: 119
  LOUISIANA: 82
  MAINE: 79
  MARYLAND: 112
  MASSACHUSETTS: 289
  MICHIGAN: 285
  MINNESOTA: 141
  MISSISSIPPI: 86
  MISSOURI: 335
  MONTANA: 10
  NEBRASKA: 98
  NEVADA: 19
  NEW BRUNSWICK: 18
  NEW HAMPSHI